# Work with `Optional` values

`X | None` — or `Optional[X]`, the same thing — is the most common kind of union in typed Python code. Handling it well is a big chunk of what type-checking actually buys you. This recipe covers the patterns.

(`Optional[X]` is an older name for exactly `X | None`. On Python 3.10+, prefer `X | None`.)

## What `Optional` means (and doesn't mean)

`Optional[X]` means "this value is either of type `X`, or it's `None`". That's it. A few things it specifically does **not** mean:

- It doesn't mean "this has a default value". Default arguments are a separate concept.
- It doesn't mean "this is missing". `None` is a value; it's just the value used as a sentinel for "no result".
- It doesn't mean "this might be absent from a dict". That's `NotRequired` in a `TypedDict`.

A function parameter `timeout: float | None = None` has both: it's optional (can be `None`) and has a default (of `None`). Those are two statements — you need both.

Common slip: `timeout: float = None`. The annotation says `float`; `None` doesn't match. Type-checker flags it. Write `float | None = None` instead.

## The narrowing pattern

When a variable's type is `X | None`, you can't use it as an `X` until you've checked for `None`. Type-checkers follow `if x is not None:`, `if x is None:`, `assert x is not None`, and early returns to narrow the type.

In [ ]:
def find_email(user_id: int) -> str | None:
    if user_id == 1:
        return "alice@example.com"
    return None

# Pattern 1: if-not-None
email = find_email(1)
if email is not None:
    # here, type-checker knows email is str
    print(email.upper())

# Pattern 2: early return
def send_welcome(user_id: int) -> None:
    email = find_email(user_id)
    if email is None:
        return                   # type-checker: past this point email is str
    print(f"Sending welcome to {email}")

send_welcome(1)
send_welcome(99)  # returns silently

Both patterns narrow the type inside (or after) the guard. Choose based on the code flow — early return reads better when the rest of the function is the "happy path".

## The assert pattern

When you're *certain* a value isn't `None` at this point in the code but the type-checker can't prove it, `assert x is not None` tells the checker (and also raises at runtime if you're wrong):

In [ ]:
def process(user_id: int, cached: dict[int, str] | None) -> None:
    if cached is None:
        cached = {}             # now it's definitely a dict
    # But if we guarded against None differently, we might need:
    assert cached is not None    # narrows to dict
    cached[user_id] = "processed"

process(1, None)

Use `assert` sparingly — it *will* raise at runtime if the assumption is wrong. For user-facing code you usually want an `if`/`raise ValueError(...)` instead, so the error message is informative.

## Default via `or`

Very common Python idiom for "use this if not None, otherwise the default":

In [ ]:
def greet(name: str | None) -> str:
    display_name = name or "stranger"
    return f"Hello, {display_name}"

print(greet("Alice"))
print(greet(None))
print(greet(""))         # empty string is also falsy!

**Watch the `or` pattern**. `None`, `""`, `0`, `[]`, and `{}` are all falsy — `name or "stranger"` returns "stranger" for all of them. If you specifically mean "if it's `None`", use `if name is None` explicitly:

```python
display_name = "stranger" if name is None else name
```

For numeric types especially, `value or default` silently replaces `0` with the default — often a bug.

## Default via ternary

Clearer than `or` for the specific-None case:

In [ ]:
def with_timeout(timeout: float | None = None) -> float:
    return timeout if timeout is not None else 30.0

print(with_timeout())
print(with_timeout(5.0))
print(with_timeout(0.0))   # 0 is correctly used as the timeout

Notice that `0.0` is passed through correctly — an `or` would have replaced it with 30.0.

## `getattr` and `.get()` with defaults

Dict `.get()` and `getattr` both accept a default:

In [ ]:
data: dict[str, str] = {"name": "Alice"}

# These all give str (not str | None):
name = data.get("name", "unknown")
email = data.get("email", "no email")

# But no-default .get() returns str | None:
maybe_phone = data.get("phone")
if maybe_phone is not None:
    print(maybe_phone)

print(name, email, maybe_phone)

The type-checker is smart about `.get()` — with a default, it infers the return as just `V`; without, it's `V | None`. So you get useful narrowing for free.

## Chained `Optional` — the walrus and `getattr`

If you're reaching into an `Optional` object for an `Optional` field, the walrus operator often reads better than nested `if` checks:

In [ ]:
class User:
    def __init__(self, email: str | None):
        self.email = email

def find_user(uid: int) -> User | None:
    return User("alice@example.com") if uid == 1 else None

# Nested with walrus:
if (user := find_user(1)) is not None and (email := user.email) is not None:
    print(email.upper())

# Equivalent without walrus:
user = find_user(1)
if user is not None and user.email is not None:
    print(user.email.upper())

Both work. The walrus version binds the name once; the non-walrus version is clearer for beginners. Pick the one your team finds readable.

## `Optional` in return types

Returning `X | None` is a promise to callers that they must handle the None case. Make the promise honestly — *don't* return `None` when you could return an empty collection instead. Empty-list-as-no-results is usually cleaner than `None`-as-no-results:

In [ ]:
# Less good: forces callers to check for None
def find_matches(pattern: str, haystack: list[str]) -> list[str] | None:
    matches = [s for s in haystack if pattern in s]
    return matches if matches else None

# Better: empty list is 'no matches'; callers can just iterate
def find_matches_better(pattern: str, haystack: list[str]) -> list[str]:
    return [s for s in haystack if pattern in s]

for m in find_matches_better("foo", ["a", "b"]):
    print(m)    # nothing to print, loop just doesn't run
print("done")

Reach for `Optional` when there's a meaningful difference between "this failed" and "this succeeded with an empty result". For most collection-returning functions, there isn't.

## Tips

- **`X | None = None`** — annotation and default are separate; both are needed.
- **Guard with `is not None`**, not `if x:` — truthy checks hit `0`, `""`, `[]` too.
- **Return empty collections**, not `None`, when the caller wants to iterate.
- **`.get(key, default)`** gives you narrowing for free on dicts.
- **Early return** for the None case when the happy path is long.